# Speakeasy board LoRA (free Colab GPU)

1. Runtime → Change runtime type → **T4 GPU**
2. Upload `training/data/train.jsonl` (from `prepare_full_dataset.py`)
3. Set `HF_USERNAME` and login
4. Run all → pushes `speakeasy-board` to Hugging Face
5. On Vercel set:
   - `HF_TOKEN=hf_...`
   - `HF_BOARD_MODEL=YOURUSER/speakeasy-board`

Every visitor then uses **your** fine-tuned board brain for free (HF Inference).

In [ ]:
!pip -q install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install datasets trl transformers huggingface_hub

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload train.jsonl
TRAIN = list(uploaded.keys())[0]
print('train file', TRAIN)

In [ ]:
from huggingface_hub import login
HF_USERNAME = "YOUR_HF_USERNAME"  # <-- change
REPO = f"{HF_USERNAME}/speakeasy-board"
login()  # paste HF write token

In [ ]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

rows = []
with open(TRAIN, encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        if 'messages' in obj:
            rows.append(obj)
        elif 'conversations' in obj:
            role = {'system':'system','human':'user','gpt':'assistant'}
            rows.append({'messages':[{'role':role.get(c['from'],'user'),'content':c['value']} for c in obj['conversations']]})
print('rows', len(rows))

max_seq = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=max_seq,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)

def formatting(example):
    return tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)

ds = Dataset.from_list(rows)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    formatting_func=formatting, max_seq_length=max_seq,
    args=TrainingArguments(
        output_dir='speakeasy-board-lora',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=200,
        learning_rate=2e-4,
        logging_steps=10,
        fp16=True,
        optim='adamw_8bit',
        report_to=[],
    ),
)
trainer.train()
model.push_to_hub(REPO, private=False)
tokenizer.push_to_hub(REPO, private=False)
print('Pushed', REPO)